# Understanding a device

When accml is working everything is nice: but reality is different: often we are faced 
with non functioning devices and then one needs to understand much more than one 
want to ... 
so here we walk you through the different layers.



## The bird eye view

accml is based on structured communication: it is like handing your drawing over to a 
skilled craftsmen. Sooner than later your piece is ready: well crafted as desired 

So lets have a look what setup we need to get accml running:
* we are now importing some standard modules: for convienence here
* some moduels of the `workbench` interface of `accml`: this is good for testing
  anything that lasts is best developed using `accml`interface
  

###  Import of modules

In [1]:
import logging

Activate the warning level below, otherwise you will see a lot of warnings

In [2]:
logging.basicConfig(level=logging.WARNING)

Standard python modules used here:

* yaml to read the yaml file containing the response data for the different quadrupoles
* json**s** to convert the data to the data model 

In [3]:
import asyncio
import jsons
import yaml
from pathlib import Path
import pprint

### importing the work bench

Now all the different symbols of

* accml
* and accml_lib

These are now imported here from the *work bench*. It is a module similar to matplotlib.pyplot: give an interface
which makes interactive work flow easier. 

* So many symbols are all found in the module *accml.work_bench.all* . These are imported as **wba**.
* Custom modules (e.g. particular machine implementations or certain solutions) are imported from custom

    * bessy2 custom module are imported as **b2**
    * what is required from the tune app is imported as **wbt**
    * ophyd async custom is imported as **wbo**
  

In [4]:
import accml.work_bench as wb
import accml.work_bench.all as wba
import accml.work_bench.lib_.custom.bessyii as b2
import accml.work_bench.app.tune as wbt
import accml.work_bench.custom.ophyd_async as wbo

/home/mfp/workspace/accml-main-lib-splitup/accml/venv-accml-tst/lib/python3.10/site-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
  warn(


matplotlib is unavailable => plotting is disabled.
To enable plotting functions, run "pip install matplotlib".
Environment variable MONGODB_URL is not defined, using default: mongodb://localhost:27017/bessyii


### Setting up accml work environment

#### The lower level layers

This setup typically only needs to be done once for the machine. Here we present you first the full setup and then we will walk step by 
step on what they do

First we need direct access to devices. We use https://blueskyproject.io/ophyd-async for this interface.

In [5]:
devices = b2.bessyii_setup(prefix=None)

On top of that we have a backend

In [6]:
backend  = wbo.OphydAsyncDeviceBackendRW(devices=devices)

Furthermore we use a "delta_" backend with a state cache
This is useful to execute measurements around the current point.

The name of the state cache can be chosen as you like. If you use more than one 
its better to be able to see what they are 

In [7]:
state_cache =wba.StateCache(name="BESSY2_OphydAsync_Dev_State_Cache")

and here the delta backend.

In [8]:
delta_backend = wbo.OphydAsyncDeltaBackendRWProxy(backend=backend, cache=state_cache)

All data is stored in a simple storage
Please note that the basic measurement execution engine 

#### setting up the measurement execution engine

we need the managers that provide allow the "glue information"

accml is designed that one has different views of the same object:

* lattice design tools (e.g. pyat) expects quadrupole with K values or to set the frequency to a cavity
* real world devices need power converters for quadrupoles or a master clock to tune the frequency of the cavity and power amplifiers to drive them.

Accml calls that different views: "design" view for the parameters and lattice elements a lattice design uses. "device" view for the real world devices. Now how to combine them.

yp (for _ellow _ages) : provides you 

Now let's load the managers and then discuss them


##### yp

In [9]:
yp, lm, ts = b2.bessyii_load_managers()

yp or "yellow pages" gives you names certain bits and pieces belong to, 
e.g. here you can see the names of all quadrupoles. We will use it further below

In [10]:
yp.get("quadrupoles")[:10]

['Q1M2D1R',
 'Q1M1T1R',
 'Q1M2T1R',
 'Q1M1D2R',
 'Q1M2D2R',
 'Q1M1T2R',
 'Q1M2T2R',
 'Q1M1D3R',
 'Q1M2D3R',
 'Q1M1T3R']

##### Liaison manager

the liason manager tells **you who speaks to whom**: e.g.

If we want to change the strength of the is magnet: we need to set the current of that particular power converter

In [11]:
lat_prop = wba.LatticeElementPropertyID("QIT6R", "main_strength")
dev_prop = lm.forward(lat_prop)
dev_prop

DevicePropertyID(device_name='PQIT6R', property='set_current')

##### Translator service

the translation service helps you to find out **what** to say

In [12]:
to = ts.get(wba.ConversionID(lattice_property_id=lat_prop, device_property_id=dev_prop))

This gives us a translation object which we can now poke to find out what we need. 

The K value of this quadrupole is roughly -2.2, which results in a current of roughly 150 A


In [13]:
to.forward(-2.2)

134.47248401770898

##### the execution environement

This is involved but as you will later on, you hardly get into contact with it, it is handled for you!

At this stage we leave the low level interface and work together with the layers above

accml comes with a very simple data storage: it stores data in memory. It follows a key-value
storage. So you have a favourite one ? Plug it in here instead!

In [14]:
storage = wba.SimpleDataStorage()

Finally we need the command rewritter as the measurement execution eninge works with structure commands.
That we will explain else where , here its enough to know that the measurement execution engine,
translates between the different views for you.

Details of the plans will be discussed elsewhere

In [15]:
rewriter = wba.CommandRewriter(liaison_manager=lm, translation_service=ts)

And finally the measurement execution engine:

In [16]:
mexec = wba.BasicMeasurementExecutionEngine(
    backend=delta_backend,
    cmd_rewriter=rewriter,
    storage=storage,
    expected_view_for_output="device",
    num_readings=2,
)

## Walking the software stack upwards

Now you have seen the whole stack: now lets see how one tracks down problems with the communication
to the outside world. 

One way is to check each level and work oneself up the stack. This approach is explained below.


### The lowest layer interfacing hardware : the ophyd async device

At the lowest level we have an ophyd async device (except when working with a simulation engine backend)
That we cover in the next section

so above you saw that we set up the devices. This needs a few more methods so that one can ask it what it knows about.

For the time being: we just go inside and look directly to the devices stored

In [17]:
devices._devices.keys()

dict_keys(['quadrupole_pcs', 'master_clock', 'tune', 'steerer_pcs', 'orbit', 'Q4PT7R', 'Q4PT5R', 'Q4PD1R', 'Q3PD2R', 'Q3PT2R', 'Q5PT7R', 'Q3P2T8R', 'Q3P2T6R', 'Q4PD7R', 'Q4PD2R', 'Q1PTR', 'Q3PT5R', 'Q3P2T1R', 'Q5PT2R', 'Q4PT2R', 'Q1PDR', 'Q3PD5R', 'Q3PD3R', 'Q3P1T6R', 'Q3PT3R', 'Q5P2T6R', 'Q4PD5R', 'Q3P1T8R', 'Q5P2T8R', 'Q2PTR', 'Q4PD3R', 'Q4PD8R', 'Q4P2T6R', 'Q5P1T8R', 'Q4PD4R', 'Q3PD7R', 'Q3PD6R', 'Q3PD8R', 'Q5PT3R', 'Q5P2T1R', 'Q4P2T1R', 'Q5PT5R', 'Q4P1T8R', 'Q5PT4R', 'Q3PT4R', 'Q4PT3R', 'Q3PD1R', 'Q5P1T1R', 'Q4P2T8R', 'Q4PD6R', 'Q3P1T1R', 'Q3PT7R', 'Q2PDR', 'Q4P1T6R', 'Q5P1T6R', 'PQIT6R', 'Q3PD4R', 'Q4PT4R', 'Q4P1T1R', 'mc-frequency', 'VS2P1T2R', 'VS3P1T2R', 'VS4P1T2R', 'VS4P2T2R'])

Now lets get then tune device:

In [18]:
tune = devices.get("tune")

and see if we can connect to it

In [19]:
await tune.connect()

Why the await? The code is "asynchronous" as much as the control systems. In notebooks we can use it directly as 
each of its cells is executed asynchronously

If that fails inspect the error meassages, but we already know its a communication problem

Such a device should be readable so lets get its answer

In [20]:
await tune.read()

{'tune-transversal-x-sig': {'value': 1064.7277346688068,
  'timestamp': 1769932486.852142,
  'alarm_severity': 0},
 'tune-transversal-y-sig': {'value': 855.9983050437191,
  'timestamp': 1769932486.852625,
  'alarm_severity': 0},
 'tune-transversal': {'value': {'x': 1064.7277346688068,
   'y': 855.9983050437191},
  'timestamp': 1769932486.8523836,
  'alarm_severity': 0}}

We can also only get the value of its base signals


In [21]:
await tune.transversal.x.sig.get_value()

1064.7277346688068

The tune device does a bit more: it structures the data

Now lets handle a settable device

In [22]:
pc = devices.get("Q3PT2R")

Let's connect and read its value

In [23]:
await pc.connect()
await pc.read()

{'Q3PT2R-readback': {'value': 222.93813060609574,
  'timestamp': 1769931389.070455,
  'alarm_severity': 0},
 'Q3PT2R-precision': {'value': 2,
  'timestamp': 1769931389.070455,
  'alarm_severity': 0},
 'Q3PT2R-set_current': {'value': 222.93813060609574,
  'timestamp': 1769931442.852106,
  'alarm_severity': 0},
 'Q3PT2R-units': {'value': '',
  'timestamp': 1769931389.070455,
  'alarm_severity': 0}}

Now lets change it a little

before we store its set point

In [25]:
ref = await pc.setpoint.get_value()
ref

222.93813060609574

In [26]:
await pc.set(ref+.1)

lets see its state

In [27]:
await pc.read()

{'Q3PT2R-readback': {'value': 223.03813060609573,
  'timestamp': 1769932503.302651,
  'alarm_severity': 0},
 'Q3PT2R-precision': {'value': 2,
  'timestamp': 1769932503.302651,
  'alarm_severity': 0},
 'Q3PT2R-set_current': {'value': 223.03813060609573,
  'timestamp': 1769932503.298216,
  'alarm_severity': 0},
 'Q3PT2R-units': {'value': '',
  'timestamp': 1769932503.302651,
  'alarm_severity': 0}}

And set it back: so that the twin (or the machine is not changed to much)

In [28]:
await pc.set(ref+.1)

### Communicating with ophyd async: the backend

We saw already that we have a backend that the measurement execution engine uses.

It is currently investigated if this extra layer is needed. For the time being its here. Why?

Because we have to deal with different views:
* the design view: how one describes an acccelerator to beam dynamics tracking codes
* the device view: the parameter set one uses on real world devices.

A backend implements (typcially) only one view

In [29]:
backend

and it tells you wich view it understands

In [30]:
backend.get_natural_view_name()

'device'

Lets construct the simulation backend for comparions: BESSY II demands to be handled a lattice reference file.
Should it use a default if no filename is given? Perhaps a good idea. We investigate that.
For the time being you need to declare where it is stored

In [31]:
data_dir = Path(__name__).absolute().parent.parent.parent

In [32]:
simulation_backend = b2.bessyii_simulator_backend(data_dir  / "05_reference_data" / "bessy2_storage_ring_reflat.json")

In [33]:
simulation_backend

DeltaBackendRWProxy(backend=SimulatorBackend(name=BESSYII_on_PyAT, acc=PyATAcceleratorSimulator(at_lattice=Lattice(<1093 elements>, name='BESSY II storage ring', energy=1718500000.0, particle=Particle('relativistic'), periodicity=1, harmonic_number=400, beam_current=0.0, nbunch=1, cavpts='CAV*'))), cache=StateCache(name=BESSYII_on_PyAT_delta_state_cache))

So you can see the simulation backend understands design

In [34]:
simulation_backend.get_natural_view_name()

'design'

You saw the command rewriter above: The measurement engine receives commands which are valid in a certain
view. It will then use the coman rewriter to rewrite them to to what the back end understands.


Now lets get some data

In [36]:
await backend.trigger("tune", "transversal")
await backend.read("tune", "transversal")

Tune(x=1064.4558342287924, y=856.9968512257008)

That can fail as a request to trigger and read does not automatically connect to the device. 
Should that be changed and a connection automatically made if none is there ? 

The measurement execution engine does it for you ... thanks to structured commands it can do it
So it will only connect to the devices you are actually using.

How we know whhat the backend understands? 
The liaison manager can help , if we look into it

The inverse look up table tells you which combination it knows .. uncomment the code below to see it all

In [44]:
# pprint.pprint(
#     list(lm.inverse_lut.keys())
# );


here some selections

In [47]:
[dev_p for dev_p in lm.inverse_lut if dev_p.device_name in ("Q1PDR" , "tune")]

[DevicePropertyID(device_name='Q1PDR', property='set_current'),
 DevicePropertyID(device_name='Q1PDR', property='delta_set_current'),
 DevicePropertyID(device_name='tune', property='transversal')]

so here we can see how the tune is called

Lets do the same for the power converter
We still need to connect it manually

In [49]:
pc = devices.get("Q1PDR")
await pc.connect()

In [52]:
await backend.trigger("Q1PDR", "set_current")
r = await backend.read("Q1PDR", "set_current")
r

{'Q1PDR-set_current': {'value': 216.17167046977923,
  'timestamp': 1769886473.876684,
  'alarm_severity': 0}}

In [53]:
ref_value = r["Q1PDR-set_current"]["value"]

Now lets set it again

In [54]:
await backend.set("Q1PDR", "set_current", ref_value+.1)

In [55]:
await backend.trigger("Q1PDR", "set_current")
r = await backend.read("Q1PDR", "set_current")
r

{'Q1PDR-set_current': {'value': 216.27167046977922,
  'timestamp': 1769933223.293327,
  'alarm_severity': 0}}

and let it set back again

In [56]:
await backend.set("Q1PDR", "set_current", ref_value+.1)

### Excursus on backends: state cache and delta backend for analysis

### How the measurement engine works with the backend

##